# InsureAI — Module 1: Machine Learning (Tabular Data)
### Capstone Project: End-to-End AI Assistant for Insurance
**Author:** InsureAI Team | **Domain:** Health & Auto Insurance Analytics

---
### Business Problem & Objectives:
1. **Part A (Regression - Premium Prediction):** Given policyholder demographic and health factors, predict the expected annual premium (`charges` / `annual_premium_inr`) with high accuracy (Target: $R^2 \ge 0.75$).
2. **Part B (Classification - Fraud Detection):** Identify suspicious auto insurance claims to reduce fraudulent payouts while handling severe class imbalance (Target: Minority F1 $\ge 0.70$).

### Workflow:
- Data loading and Exploratory Data Analysis (EDA)
- Outlier detection and missing value handling
- Categorical encoding (Label & One-Hot Encoding) & feature scaling
- Strict train/test split to prevent data leakage
- Handling class imbalance using SMOTE on the training split only
- Training and comparative evaluation across multiple algorithms
- Saving production model artifacts (`.pkl`) for Streamlit integration


In [ ]:
# 1. Setup & Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries imported successfully.')


## Part A: Premium Prediction (Regression)
Let's load the dataset, inspect column distributions, check correlations, and engineer features.


In [ ]:
# Load Premium Dataset
df_prem = pd.read_csv('../data/insurance_premium.csv')
print(f'Dataset Shape: {df_prem.shape}')
df_prem.head()


In [ ]:
# EDA: Check summary statistics & missing values
print('Missing Values:')
print(df_prem.isnull().sum())
print('\nSummary Statistics:')
display(df_prem.describe())


In [ ]:
# EDA Visualizations: Premium Distribution & Smoker Impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_prem['annual_premium_inr'], kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('Annual Premium Distribution (INR)')
axes[0].set_xlabel('Premium (INR)')

sns.boxplot(x='smoker', y='annual_premium_inr', data=df_prem, ax=axes[1], palette='Set2')
axes[1].set_title('Impact of Smoking Status on Premium')
axes[1].set_ylabel('Annual Premium (INR)')
plt.tight_layout()
plt.show()


In [ ]:
# Preprocessing Pipeline for Regression
df_p = df_prem.drop(columns=['customer_id']).copy()
df_p['bmi'] = df_p['bmi'].fillna(df_p['bmi'].median())
df_p['annual_income_inr'] = df_p['annual_income_inr'].fillna(df_p['annual_income_inr'].median())

cat_cols = ['gender', 'smoker', 'region', 'occupation', 'exercise_frequency', 'alcohol_consumption', 'medical_history', 'family_medical_history']
num_cols = ['age', 'bmi', 'children', 'annual_income_inr']

# Train/Test Split (80/20) BEFORE fitting transformers to avoid data leakage
X = df_p.drop(columns=['annual_premium_inr'])
y = df_p['annual_premium_inr'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

encoder_p = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_train_cat = encoder_p.fit_transform(X_train[cat_cols])
X_test_cat = encoder_p.transform(X_test[cat_cols])

scaler_p = StandardScaler()
X_train_num = scaler_p.fit_transform(X_train[num_cols])
X_test_num = scaler_p.transform(X_test[num_cols])

X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final = np.hstack([X_test_num, X_test_cat])
print(f'Train shape: {X_train_final.shape}, Test shape: {X_test_final.shape}')


In [ ]:
# Train at least 3 Regression Models and Compare
reg_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.06, random_state=42, n_jobs=-1)
}

reg_summary = []
for name, model in reg_models.items():
    model.fit(X_train_final, y_train)
    preds = model.predict(X_test_final)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    reg_summary.append({'Model': name, 'R2 Score': round(r2, 4), 'MAE (INR)': round(mae, 2), 'RMSE (INR)': round(rmse, 2)})

df_reg_results = pd.DataFrame(reg_summary)
display(df_reg_results)


## Part B: Fraud Detection (Classification)
Auto insurance fraud accounts for billions in annual losses. We build a classification engine using SMOTE to handle class imbalance.


In [ ]:
# Load Fraud Claims Dataset
df_fraud = pd.read_csv('../data/insurance_claims.csv')
print(f'Fraud claims dataset shape: {df_fraud.shape}')
print('Target Distribution:')
print(df_fraud['fraud_reported'].value_counts(normalize=True))


In [ ]:
# Preprocessing & Handling Missing Values
df_f = df_fraud.drop(columns=['claim_id']).copy()
df_f['days_to_report'] = df_f['days_to_report'].fillna(df_f['days_to_report'].median())
df_f['fraud_reported'] = (df_f['fraud_reported'] == 'Y').astype(int)

f_cat_cols = ['gender', 'policy_type', 'incident_type', 'incident_severity', 'police_report_filed', 'documents_complete', 'claim_channel', 'income_bracket', 'claim_location']
f_num_cols = ['customer_age', 'policy_tenure_years', 'annual_premium_inr', 'claim_amount_inr', 'days_to_report', 'witnesses', 'num_past_claims_3yrs']

X_f = df_f.drop(columns=['fraud_reported'])
y_f = df_f['fraud_reported'].values

X_f_tr, X_f_te, y_f_tr, y_f_te = train_test_split(X_f, y_f, test_size=0.2, random_state=42, stratify=y_f)

enc_f = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_f_tr_cat = enc_f.fit_transform(X_f_tr[f_cat_cols])
X_f_te_cat = enc_f.transform(X_f_te[f_cat_cols])

scaler_f = StandardScaler()
X_f_tr_num = scaler_f.fit_transform(X_f_tr[f_num_cols])
X_f_te_num = scaler_f.transform(X_f_te[f_num_cols])

X_f_tr_pre = np.hstack([X_f_tr_num, X_f_tr_cat])
X_f_te_fin = np.hstack([X_f_te_num, X_f_te_cat])

# Apply SMOTE on TRAINING SPLIT ONLY
print(f'Before SMOTE: {np.bincount(y_f_tr)}')
smote = SMOTE(random_state=42)
X_f_tr_sm, y_f_tr_sm = smote.fit_resample(X_f_tr_pre, y_f_tr)
print(f'After SMOTE:  {np.bincount(y_f_tr_sm)}')


In [ ]:
# Train at least 3 Classification Models and Compare
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, random_state=42, n_jobs=-1)
}

clf_summary = []
opt_thresh = 0.42
for name, model in clf_models.items():
    model.fit(X_f_tr_sm, y_f_tr_sm)
    prob = model.predict_proba(X_f_te_fin)[:, 1]
    pred = (prob >= opt_thresh).astype(int) if name in ['XGBoost', 'Random Forest'] else (prob >= 0.5).astype(int)
    acc = accuracy_score(y_f_te, pred)
    prec = precision_score(y_f_te, pred, zero_division=0)
    rec = recall_score(y_f_te, pred, zero_division=0)
    f1 = f1_score(y_f_te, pred, zero_division=0)
    auc = roc_auc_score(y_f_te, prob)
    clf_summary.append({'Model': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4), 'Recall': round(rec, 4), 'F1-Score': round(f1, 4), 'ROC-AUC': round(auc, 4)})

df_clf_results = pd.DataFrame(clf_summary)
display(df_clf_results)


In [ ]:
# Confusion Matrix for the Best Classifier
best_clf = clf_models['XGBoost']
cm = confusion_matrix(y_f_te, (best_clf.predict_proba(X_f_te_fin)[:, 1] >= opt_thresh).astype(int))
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix — XGBoost Fraud Classifier')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.show()
